# M16 · 11 — B2: stage B post-training and evaluation

> **⚠ NOT EXECUTED.** Stage B is out of M16's scope (decision 0025, amendment of 2026-11-15).
> This notebook was written so the work can be resumed later. It has **never been run**, and there
> are **no results** in it. Do not read anything here as a measurement.

B2 repeats A3 (notebooks 06–08) on the stage B model, with what A3 taught:
- general SFT 1 epoch; task SFT **1 epoch** (3 overfit at 110M), replay ~15 % of the loss tokens;
- DPO pairs from `llm.dpo_pairs` on the stage B SFT model, reduce prompts sampled ≥ 4 times;
- relevance needs more than DPO on "disagrees with teacher" pairs: at 110M both SFT and DPO models
  produced content-independent score shapes (notebook 07). A per-block prompt or a scoring head
  would be the first thing to try;
- gates with `llm.evaluate` on the same 49 test episodes, then **C3**: blind review per task.

In [ ]:
# Commands, in order (each one is what A3 ran, with stage B paths):
STEPS = '''
llm-train start configs\\b2\\sft_general.yaml        # init_from runs/stage-b-330m/final_model.pt
llm-train start configs\\b2\\sft_tasks.yaml          # 1 epoch, init_from the general SFT
python -m llm.export_gguf runs/b2-sft-tasks/final_model.pt tokenizer/tokenizer.json <app>/models/llm/yts-own-330m-sft/yts-own-330m-sft.gguf
python -m llm.dpo_pairs build --own-dir <app>/models/llm/yts-own-330m-sft --per-task 1000
python -m llm.dpo_pairs build --own-dir <app>/models/llm/yts-own-330m-sft --tasks shownotes_reduce --repeats 3 --out .../pairs_reduce_x3.jsonl
python -m llm.dpo build --out data/sft/dpo-b .../pairs.jsonl .../pairs_reduce_x3.jsonl
python -m llm.dpo ref data/sft/dpo-b runs/b2-sft-tasks/final_model.pt
llm-train start configs\\b2\\dpo.yaml
python -m llm.evaluate run --model own --label own-b-dpo --own-dir <app>/models/llm/yts-own-330m-dpo
python -m llm.evaluate gates --label own-b-dpo
'''
print(STEPS)

In [ ]:
import json
from pathlib import Path
from llm import HOME
RUNS, SFT, EVAL = HOME / 'runs', HOME / 'data' / 'sft', HOME / 'eval'
def load(p): return json.loads(Path(p).read_text())
def metrics(run): return [json.loads(l) for l in (RUNS / run / 'metrics.jsonl').read_text().splitlines()]
from llm.evaluate import gates
for label in ('own-dpo', 'own-b-sft', 'own-b-dpo'):   # stage A DPO model as the baseline
    if (EVAL / label).exists():
        print(json.dumps(gates(label), indent=1))